The program can be used to test different embeddings of text, metrics, and LLM responses

## A compact version of RAG test for deployment in HF

In [ ]:
from dotenv import load_dotenv
from openai import OpenAI
from anthropic import Anthropic
import os
#import numpy as np
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import gradio as gr
from ordered_set import OrderedSet
#from huggingface_hub.utils import disable_progress_bars
#disable_progress_bars()
from transformers.utils import logging as hf_logging
hf_logging.disable_progress_bar()
#import json
#import requests
#import time
#import textwrap
#import tiktoken
#import bm25s 
#import pandas as pd
# this needs running ollama server (not practical for HF), it is also depreciated:
#from langchain_community.embeddings import OllamaEmbeddings

import RAG_lib as rlib


In [18]:
if 1:
    # reload rlib if it was changed
    from importlib import reload
    reload(rlib);

## General variables and default options

In [19]:
# Variables
var_dict = {}
# ~ 1000 words or tokens (~ 2 pages) is a good chunk size for scientific texts
# ~ 500 words (~ 1 page) may be OK too
var_dict['chunk_size'] = 1024 # 512
# overlap between chunks - useful if context is at the boundary of a chunk
var_dict['chunk_overlap'] = 200
var_dict['chunk_overlap_BM25'] = 0 # overlap for BM25 RAG
#chunk_overlap_dict = {'BM25':0} # custom overlap for different RAGs
var_dict['ktop'] =  5 # if 0, then RAG is not used
var_dict['print_costs'] = True # print token usage and costs for LLMs

# BM25, vector
#RAG_representation = 'tokens' # 'tokens', 'vectors'
#RAG_metric = 'overlap_weighted'
RAG_options = {}
RAG_options['tokens'] = {}
RAG_options['vectors'] = {}

# use a dictionary to query metrics for calculating similarity of embeddings
metric_dict = {'cosine': lambda xs, ys: cosine_similarity(xs, ys)[0]}

# an example of a vector model
#RAG_options['vectors']['ollama:nomic-embed-text_cos']  = {}
#RAG_options['vectors']['ollama:nomic-embed-text_cos']['embedding']  = \
#    OllamaEmbeddings(model="nomic-embed-text").embed_documents
#RAG_options['vectors']['ollama:nomic-embed-text_cos']['metric'] = 'cosine'

# an example of a vector model
RAG_options['vectors']['transformer:all-MiniLM-L6-v2']  = {}
RAG_options['vectors']['transformer:all-MiniLM-L6-v2']['embedding']  = \
    SentenceTransformer('all-MiniLM-L6-v2').encode
RAG_options['vectors']['transformer:all-MiniLM-L6-v2']['metric'] = 'cosine'


# an example of a token model: words
RAG_options['tokens']['words'] = {}
RAG_options['tokens']['words']['embedding'] = \
    rlib.token_freq_df_embed_documents(tokenizer=rlib.get_words)
RAG_options['tokens']['words']['metric'] = 'token_similarity_weighted'

# an example of a token model: words + BM25
RAG_options['tokens']['BM25'] = {}
RAG_options['tokens']['BM25']['embedding'] = rlib.bm25_embedding
RAG_options['tokens']['BM25']['metric'] = 'BM25_default'

metric_dict['BM25_default'] = rlib.bm25_metric

RAG_dict = {'vectors':['transformer:all-MiniLM-L6-v2'],
             'tokens':['words', 'BM25']
             }

var_dict['RAG_dict'] = RAG_dict

var_dict['RAGs'] = OrderedSet()
for RAGs in RAG_dict.values():
    var_dict['RAGs'] = var_dict['RAGs'].union(RAGs)

# LLM options
#LLMs = ['gpt-5-mini', 'gpt-5-nano', 'llama3.2', "claude-sonnet-5-5", "gemini-2.5-flash"]
LLM_dict = {'openai':['gpt-5-mini', 'gpt-5-nano'],
             'anthropic':["claude-sonnet-5-5"],
             'google':["gemini-2.5-flash"],
             'hf':[]}

var_dict['LLM_dict'] = LLM_dict

million = 1.e6
var_dict['token_cost_dict'] = \
    {'gpt-5-mini': {'input': 0.25/million, 'output': 2./million},
    'gpt-5-nano': {'input': 0.05/million, 'output': 0.4/million},
    'claude-sonnet-5-5': {'input': 2./million, 'output': 10./million},
    'gemini-2.5-flash': {'input': 0.3/million, 'output': 2.5/million}
    }




var_dict['LLMs'] = OrderedSet()
for LLMs in LLM_dict.values():
    var_dict['LLMs'] = var_dict['LLMs'].union(LLMs)

#hf_models = ["meta-llama/Llama-3.2-3B:auto", "openai/gpt-oss-20b:auto", "Qwen/Qwen3.5-9B:auto"]

var_dict['RAG'] = 'transformer:all-MiniLM-L6-v2' #'words'
var_dict['LLM'] = 'Qwen/Qwen3-0.6B:featherless-ai' #'claude-sonnet-5-5'

## Meta data, e.g., cost information

## Environment variables

In [20]:
# load environment variables from .env
load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')
hf_api_key = os.getenv('HF_TOKEN')

if 1:
    # test if api keys exist
    if openai_api_key:
        print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
    else:
        print("OpenAI API Key not set")
        
    if anthropic_api_key:
        print(f"Anthropic API Key exists and begins {anthropic_api_key[:7]}")
    else:
        print("Anthropic API Key not set (and this is optional)")
        
    if google_api_key:
        print(f"Google API Key exists and begins {google_api_key[:2]}")
    else:
        print("Google API Key not set (and this is optional)")

    if hf_api_key:
        print(f"HF API Key exists and begins {hf_api_key[:2]}")
    else:
        print("HF API Key not set (and this is optional)")

OpenAI API Key exists and begins sk-proj-
Anthropic API Key exists and begins sk-ant-
Google API Key exists and begins AI
HF API Key exists and begins hf


In [21]:


def openai_bot(var_dict):
    model = var_dict['LLM']
    full_response = var_dict.get('full_response', 0)
    bot = OpenAI()
    def chatbot(messages, full_response=full_response):
        response = bot.chat.completions.create(model=model, messages=messages)
        response_dict = {'text': response.choices[0].message.content}
        usage = response.usage
        response_dict['usage'] = {'prompt_tokens': usage.prompt_tokens,
                                  'completion_tokens': usage.completion_tokens,
                                  'reasoning_tokens': None}
        if hasattr(usage, 'completion_tokens_details') and usage.completion_tokens_details:
            reasoning_tokens = usage.completion_tokens_details.reasoning_tokens
            response_dict['usage']['reasoning_tokens'] = reasoning_tokens
        if full_response:
            return response
        else:
            return response_dict
    return chatbot

def google_bot(var_dict):
    model = var_dict['LLM']
    full_response = var_dict.get('full_response', 0)
    bot = OpenAI(api_key=google_api_key, 
                 base_url="https://generativelanguage.googleapis.com/v1beta/openai/")
    def chatbot(messages, full_response=full_response):
        response = bot.chat.completions.create(model=model, messages=messages)
        response_dict = {'text': response.choices[0].message.content}
        usage = response.usage
        response_dict['usage'] = {'prompt_tokens': usage.prompt_tokens,
                                  'completion_tokens': usage.completion_tokens,
                                  'reasoning_tokens': None}
        if hasattr(usage, 'completion_tokens_details') and usage.completion_tokens_details:
            reasoning_tokens = usage.completion_tokens_details.reasoning_tokens
            response_dict['usage']['reasoning_tokens'] = reasoning_tokens
        return response_dict
    return chatbot


def anthropic_bot(var_dict):
    model = var_dict['LLM']
    full_response = var_dict.get('full_response', 0)
    bot = Anthropic()
    def chatbot(messages, full_response=full_response):
        response = bot.messages.create(model=model, messages=messages, max_tokens=1000)
        #print('Anthropic response:', response)
        for r in response.content:
            # return the first text response, skip thinking blocks
            if r.type == 'text':
                response_dict = {'text': r.text}
                break
        usage = response.usage
        # placeholder for usage for anthropic
        response_dict['usage'] = {'prompt_tokens': usage.input_tokens,
                                  'completion_tokens': usage.output_tokens,
                                  'reasoning_tokens': None}
        if hasattr(usage, 'output_tokens_details') and usage.output_tokens_details.thinking_tokens:
            reasoning_tokens = usage.output_tokens_details.thinking_tokens
            response_dict['usage']['reasoning_tokens'] = reasoning_tokens
        return response_dict
    return chatbot



def hf_bot(var_dict):
    model = var_dict['LLM']
    full_response = var_dict.get('full_response', 0)
    bot = OpenAI(
        base_url="https://router.huggingface.co/v1",
        api_key=hf_api_key
        )

    def chatbot(messages, full_response=full_response):
        response = bot.chat.completions.create(model=model, messages=messages)
        response = bot.chat.completions.create(
                model=model,  # ":any other providers" is required
                messages=messages)
        response_text = str(response.choices[0].message.content)
        response_dict = {'text': response_text}
        usage = response.usage
        response_dict['usage'] = {'prompt_tokens': usage.prompt_tokens,
                                  'completion_tokens': usage.completion_tokens,
                                  'reasoning_tokens': None}
        if hasattr(usage, 'completion_tokens_details') and usage.completion_tokens_details:
            reasoning_tokens = usage.completion_tokens_details.reasoning_tokens
            response_dict['usage']['reasoning_tokens'] = reasoning_tokens
        return response_dict
    return chatbot


def get_bot(var_dict):
    model = var_dict['LLM']
    LLM_dict = var_dict['LLM_dict']
    if model in LLM_dict.get('openai', []):
        return openai_bot(var_dict)
    elif model in LLM_dict.get('anthropic', []):
        return anthropic_bot(var_dict)
    elif model in LLM_dict.get('google', []):
        return google_bot(var_dict)
    elif model in LLM_dict.get('hf', []):
        return hf_bot(var_dict)
    else:
        return None



## RAG preprocessing

In [22]:
# converte pdf to text and save (if not yet converted)
# generalize later for on-demand additions to RAG
fn0 = 'LAT_FBs-1407.7905v1' # Fermi LAT paper about the Fermi bubbles
pdf_fn = 'RAG_data/%s.pdf' % fn0
txt_fn = 'RAG_data/%s.txt' % fn0
rewrite_txt = False
if not os.path.exists(txt_fn) or rewrite_txt:
    text = rlib.pdf2text(pdf_fn)
    with open(txt_fn, "w", encoding="utf-8") as f:
        f.write(text)
else:
    text = open(txt_fn, "r", encoding="utf-8").read()
var_dict['text'] = text

# calculate the word frequency in the corpus
text_dfs = RAG_options['tokens']['words']['embedding']([text])
word_freq_df = rlib.add_dfs(text_dfs, col_name='frequency')
    

# use the word frequencies as inverse weights
metric_dict['token_similarity_weighted'] = \
    rlib.token_similarity_weighted(word_freq_df)


## Example questions

In [23]:
qfn = 'RAG_data/FB_example_questions.txt'
qs = open(qfn, 'r').readlines()
qs = [q.strip() for q in qs if not q.startswith('#')]

In [24]:

test = 0
if test:
    model_name = 'claude-sonnet-5-5'
    #q = 'How many prime numbers are there between 1 and 100?'
    q = 'size fermi bubbles'
    test_messages = [{"role": "user", "content": q}]
    test_messages.append({"role": "user", "content": RAG_response(q, var_dict)})
    bot = Anthropic()
    response = bot.messages.create(model=model, messages=test_messages, max_tokens=1000)
    
    


In [25]:
if test:
    for i, r in enumerate(response.content):
        print(i, r.type)
        if r.type == 'text':
            response_dict = {'text': r.text}
            break
    print(response_dict)
    response.usage

In [26]:
if test:
    model = 'claude-sonnet-5-5'
    print(test_messages)
    bot = get_bot(var_dict)
    response = bot(messages=test_messages)
    print(response)

In [27]:
#q = 'What is the capital of France?'
#q = 'How many prime numbers are there between 1 and 100?'
q = 'size of fermi bubbles'

test_messages = [{"role": "user", "content": q}]
LLM_dict['hf'] = ["meta-llama/Llama-3.2-1B-Instruct:featherless-ai", # 0
                    "meta-llama/Llama-3.2-3B-Instruct:featherless-ai", # 1
                    "Qwen/Qwen3-0.6B:featherless-ai", # 2
                    "google/gemma-3-4b-it:deepinfra", # 3
                    "Qwen/Qwen3.5-9B:together", # 4
                    "openai/gpt-oss-20b:groq", # 5
                    ]

var_dict['LLMs'] = var_dict['LLMs'].union(LLM_dict['hf'])

model = LLM_dict['hf'][5]
print(q)
print(model)




size of fermi bubbles
openai/gpt-oss-20b:groq


In [28]:
bot = get_bot(var_dict)
response = bot(messages=test_messages)
print('Answer:', response['text'])
print('Usage:', response['usage'])

Answer: The size of a Fermi bubble is influenced by the number of electrons and the temperature. Here's a structured breakdown:

1. **Fermi Bubbles and Superconductivity**: Fermi bubbles are formed when electrons are arranged in a larger Fermi surface due to the superconducting nature. This occurs when the electrons occupy more states at higher temperatures.

2. **Size Dependence**:
   - **Number of Electrons**: The size of the Fermi surface is proportional to the square of the number of electrons. More electrons lead to a larger Fermi surface.
   - **Temperature**: At higher temperatures, the electrons are more mobile, allowing the Fermi surface to expand. However, once the temperature drops below the critical temperature, the superconductor transitions to non-superconducting behavior, reducing the size of the Fermi surface.

3. **Theoretical Model**: The size can be described by the density of states (DOS) in the superconductor. The area of the Fermi surface is proportional to the sq

### Test RAGs with different questions

In [29]:
def get_type(model, type_dict):
    for key, models in type_dict.items():
        if model in models:
            return key
    return None


def RAG_response(query, var_dict):
    chunks = var_dict['chunks']
    RAG = var_dict['RAG']
    RAG_dict = var_dict['RAG_dict']
    ktop = var_dict['ktop']
    chunk_overlap = var_dict['chunk_overlap']
    chunk_size = var_dict['chunk_size']
    text = var_dict['text']
    
    if RAG == 'BM25':
        chunk_overlap = var_dict['chunk_overlap_BM25']

    RAG_type = get_type(RAG, RAG_dict)
        
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )

    chunks = splitter.split_text(text)

    embedding = RAG_options[RAG_type][RAG]['embedding']
    metric_name = RAG_options[RAG_type][RAG]['metric']
    metric = metric_dict[metric_name]
    emb_chunks = embedding(chunks)
    emb_qs = embedding([query])

    scores = metric(emb_qs, emb_chunks)
    inds = np.argsort(scores)[::-1][:ktop]

    message = ''
    for i, ind in enumerate(inds):
        message += f'{i + 1}: {chunks[ind]}\n\n'
    return message



    


In [30]:
#RAG = 'ollama:nomic-embed-text_cos'
#RAG_type = rlib.get_type(RAG, RAG_dict)
chunk_overlap = var_dict['chunk_overlap']
if var_dict['RAG'] == 'BM25':
    chunk_overlap = var_dict['chunk_overlap_BM25']


## Split text in chunks

In [31]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=var_dict['chunk_size'],
    chunk_overlap=var_dict['chunk_overlap'],
    length_function=len,
    separators=["\n\n", "\n", " ", ""]
)

chunks = splitter.split_text(text)
var_dict['chunks'] = chunks
print('The number of chunks is', len(chunks))

NameError: name 'RecursiveCharacterTextSplitter' is not defined

In [ ]:
# test new RAG embedding
if 0:
    embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
    emb_chunks = embedding_model.encode(var_dict['chunks'], show_progress_bar=True)
    q = qs[1]
    emb_qs = embedding_model.encode([q], show_progress_bar=False)
    metric_name = 'cosine'
    metric = metric_dict[metric_name]
    scores = metric(emb_qs, emb_chunks)
    ktop = var_dict['ktop']
    inds = np.argsort(scores)[::-1][:ktop]

    message = ''
    for i, ind in enumerate(inds):
        message += f'{i + 1}: {chunks[ind]}\n\n'

    print('Chunk length:', len(var_dict['chunks']), '\nEmbedding shape:', emb_chunks.shape)
    print('Top scores:', scores[inds])
    print('Query:', q)
    print('Responses:')
    print(message)

Batches: 100%|██████████| 6/6 [00:03<00:00,  1.92it/s]

Chunk length: 186 
Embedding shape: (186, 384)
Top scores: [0.6216928  0.6017631  0.59425557 0.5704787  0.5684706 ]
Query: What is the distance to the Fermi bubbles in kpc, assuming that they are above the Galactic center?
Responses:
1: from the region near the GC, it is not certain that they are associated with the GC region rather than
from a region closer to the Earth. There are nevertheless several indirect arguments that the Fermi
bubbles were created by a phenomenon in or around the Galactic center. First, a symmetry argument:
the bubbles appear to be directly above and below the GC. Second, the hard energy spectrum and the
sharp edges favor a transient nature for the bubbles. Locally the bubbles can be produced by a SN
explosion, but in this case one expects to find strong synchrotron emission, while the bubbles, assuming
that the association with the WMAP andPlanck haze is correct, have very weak synchrotron emission
at high latitudes and relatively strong synchrotron emission 

In [ ]:
# quick check of RAG response
q = qs[1]
print(RAG_response(q, var_dict))

1: from the region near the GC, it is not certain that they are associated with the GC region rather than
from a region closer to the Earth. There are nevertheless several indirect arguments that the Fermi
bubbles were created by a phenomenon in or around the Galactic center. First, a symmetry argument:
the bubbles appear to be directly above and below the GC. Second, the hard energy spectrum and the
sharp edges favor a transient nature for the bubbles. Locally the bubbles can be produced by a SN
explosion, but in this case one expects to find strong synchrotron emission, while the bubbles, assuming
that the association with the WMAP andPlanck haze is correct, have very weak synchrotron emission
at high latitudes and relatively strong synchrotron emission at lower latitudes, which can be naturally
explained by a decreasing magnetic field at larger distances from the Galactic plane. Third, a series
ofSuzaku X-ray observations across the edge of the bubbles (Kataoka et al. 2013) reveal a

In [ ]:
vars = ['LLM', 'RAG', 'chunk_size', 'chunk_overlap', 'chunk_overlap_BM25', 'ktop', 'print_costs']
special_commands = ['/help', '/LLMs', '/RAGs', '/vars']
var_dict['vars'] = vars
var_dict['special_commands'] = special_commands


In [ ]:

def chat(message, history, var_dict=var_dict):
    messages = history + [message]
    if message.startswith('/'):
        response = rlib.execute(message, var_dict=var_dict)
        return response
    if message.find('@') == -1:
        message = '@RAG+LLM: ' + message

    target = message.split(' ')[0]
    query = message.replace(target, '').strip()
    response = ''
    costs = None
    if var_dict['ktop'] > 0 and message.lower().find('rag') > -1:
        # determine RAG response (or context if LLM is used)
        response += f'RAG "{var_dict["RAG"]}" top {var_dict["ktop"]} chunks '
        answer = RAG_response(query, var_dict)
    if message.lower().find('llm') > -1:
        messages = [{"role": "user", "content": query}]
        if len(response) > 0:
            response += '+ '
            messages.append({"role": "user", "content": answer})
        
        
        bot = get_bot(var_dict)
        try:
            bot_response = bot(messages)
            answer = bot_response['text'] # if LLM is used: overwrite answer with LLM response
        except Exception as e:
            bot_response = {}
            answer = f"Error: {e}"
        #print(bot_response)
        
        response += f'LLM "{var_dict["LLM"]}" '
        if var_dict.get('print_costs', False):
            usage_dict = bot_response.get('usage', {})
            costs = rlib.get_token_costs(usage_dict, var_dict)
            #print('Usage dict and costs:',usage_dict, costs)            
    response += f'response:\n\n{answer}'
    if costs is not None:
        response += f'\n\n{costs}'

    return response


In [ ]:
chat_box = gr.ChatInterface(chat, #type="messages",     
                            textbox=gr.Textbox(placeholder="type /help for available commands", 
                                               lines=1, max_lines=30))
chat_box.launch()

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.
